Listing of All Businesses
https://data.lacity.org/Administration-Finance/Listing-of-All-Businesses/r4uk-afju/about_data


## Dataset Loading & checking basic information

In [ ]:
import pandas as pd

In [ ]:
file_path = 'dataset\\Listing_of_All_Businesses_20250202.csv'
# file_url = 'https://media.githubusercontent.com/media/EricSJSU-DataScience/CS163_project/refs/heads/main/dataset/Listing_of_All_Businesses_20250202.csv'
df = pd.read_csv(file_path, 
                 dtype={"NAICS": "Int64"}, 
                 parse_dates=["LOCATION START DATE", "LOCATION END DATE"]
                 )
df.info()

In [ ]:
df['LOCATION END DATE'] = pd.to_datetime(df['LOCATION END DATE'], errors='coerce', infer_datetime_format=True)
df.info()

In [ ]:
# drop na NAICS and na Start date
df1 = df.dropna(subset=['NAICS', 'LOCATION START DATE'])

#### NAICS info

In [ ]:
file_path = 'dataset\\naics_2_clean.csv'
naics_2 = pd.read_csv(file_path)

In [ ]:
naics_dict = naics_2.set_index('Code')['Sector_Title'].to_dict()

In [ ]:
df1['NAICS_2'] = df1['NAICS'].astype(str).str[:2].astype(int)
df1['2d_title'] = df1['NAICS_2'].map(naics_dict)

### GeoGraph

In [ ]:
import folium
import geopy
import pandas as pd
import numpy as np
from folium.plugins import MarkerCluster, FastMarkerCluster 

In [ ]:
city_count_dict = df1['CITY'].value_counts().to_dict()
print(city_count_dict)
print(len(city_count_dict))

Load city_list dataset

In [ ]:
file_path = 'dataset\\city_list.csv'
city_df = pd.read_csv(file_path)
city_list = city_df['City'].to_list()

In [ ]:
# for later Geomap use
city_dict = {'City': city_list}

In [ ]:
# Function to parse coordinates
def parse_location(location_str):
    try:
        lat, lon = location_str.strip('()').split(', ')
        return float(lat), float(lon)
    except:
        return None, None

In [ ]:
# Extract latitude and longitude
df1[['latitude', 'longitude']] = df1['LOCATION'].apply(lambda x: pd.Series(parse_location(x)))

In [ ]:
# Filter invalid coordinates
df1 = df1.dropna(subset=['latitude', 'longitude'])
df1 = df1[(df1['latitude'].between(-90, 90)) & (df1['longitude'].between(-180, 180))]

In [ ]:
# Map center, manually set to LA union station parking
map_center = [34.05525, -118.23737]

# Initialize the map
business_map = folium.Map(location=map_center, zoom_start=12)

points = df1[['latitude', 'longitude']].values.tolist()
FastMarkerCluster(points).add_to(business_map)
# # Add marker clusters # too slow, use FastMarkerCluster() instead
# marker_cluster = MarkerCluster().add_to(business_map)

# cur = 0
# total = len(df1)
# # Add business markers
# for _, row in df1.iterrows():
#     cur += 1
#     folium.Marker(
#         location=[row['latitude'], row['longitude']],
#         popup=folium.Popup(f"{row['BUSINESS NAME']}<br>{row['STREET ADDRESS']}", max_width=250),
#         icon=folium.Icon(color='blue', icon='info-sign')
#     ).add_to(marker_cluster)
#     print(f'\rProcessing: {cur}/{total}  ({cur / total: .0%})\t\t', end='', flush=True)
# print(f'Process Completed!')

# Display the map
business_map

As the map shown, the coordinate may not be all correct across dataset.
More than 36 thousands coordinate not in US. 

Process Problem: using MarkerCluster, dataset content processed under 10 min, however the map does not render successful possiblely due to too many data point. using FastMarkerCluster, map successful render but there isn't label correspond with each point to show information. 

### use zip code to filter any record outside LA area

In [ ]:
file_path = 'dataset\\zip_code.csv'
zip_code = pd.read_csv(file_path)

In [ ]:
zip_code_list = zip_code['Zip_Code'].to_list()

In [ ]:
df1 = df.dropna(subset=['NAICS', 'LOCATION START DATE'])
df1['5d_zip'] = df1['ZIP CODE'].str.split('-').str[0]
df1['5d_zip_num'] = pd.to_numeric(df1['5d_zip'], errors='coerce')

In [ ]:
df2 = df1[df1['5d_zip'].isin(zip_code_list)]

In [ ]:
# Inspect the first 20 unique zip codes from df1['5d_zip']
unique_zips = df1['5d_zip'].unique()
print("Unique values in df1['5d_zip'] (first 20):")
print(unique_zips[:20])

# Check the data type of the column
print("\nData type of df1['5d_zip']:", df1['5d_zip'].dtype)

# Inspect the first few items of your zip_code_list
print("\nFirst 20 items in zip_code_list:")
print(zip_code_list[:20])

# Check the type of an item from zip_code_list
if len(zip_code_list) > 0:
    print("\nData type of first item in zip_code_list:", type(zip_code_list[0]))

# See how many zip codes in df1 match your list
intersection = set(df1['5d_zip'].unique()).intersection(set(zip_code_list))
print("\nNumber of matching zip codes:", len(intersection))
print("Matching zip codes:", intersection)


In [ ]:
zip_code_list = [str(z).zfill(5) for z in zip_code_list]

In [ ]:
df2 = df1[df1['5d_zip'].isin(zip_code_list)]
print("Filtered records:", len(df2))

**Note:** df2 should be exclude records outside of LA area 

In [ ]:
# Function to parse coordinates
def parse_location(location_str):
    try:
        lat, lon = location_str.strip('()').split(', ')
        return float(lat), float(lon)
    except:
        return None, None

In [ ]:
# Extract latitude and longitude
df2[['latitude', 'longitude']] = df2['LOCATION'].apply(lambda x: pd.Series(parse_location(x)))

In [ ]:
df2 = df2.dropna(subset=['latitude', 'longitude'])

Note: filter dataset based on coordinate range

In [ ]:
latitudes_max = 34.45
latitudes_min = 33.24
longitudes_max = -116.8
longitudes_min = -118.7

In [ ]:
df2_filtered = df2[
    (df2['latitude'] >= latitudes_min) & (df2['latitude'] <= latitudes_max) &
    (df2['longitude'] >= longitudes_min) & (df2['longitude'] <= longitudes_max)
]

In [ ]:
# Map center, manually set to LA union station parking
map_center = [34.05525, -118.23737]

# Initialize the map
business_map = folium.Map(location=map_center, zoom_start=12)

points = df2[['latitude', 'longitude']].values.tolist()
FastMarkerCluster(points).add_to(business_map)

# # Add marker clusters
# marker_cluster = MarkerCluster().add_to(business_map)

# cur = 0
# total = len(df3)
# # Add business markers
# for _, row in df3.iterrows():
#     cur += 1
#     folium.Marker(
#         location=[row['latitude'], row['longitude']],
#         popup=folium.Popup(f"{row['BUSINESS NAME']}<br>{row['STREET ADDRESS']}", max_width=250),
#         icon=folium.Icon(color='blue', icon='info-sign')
#     ).add_to(marker_cluster)
#     print(f'\rProcessing: {cur}/{total}  ({cur / total: .0%})\t\t', end='', flush=True)
# print(f'Process Completed!')

# Display the map
business_map

**Note:** try plotly express choropleth later

In [ ]:
import pandas as pd
import plotly.express as px

# Sample DataFrame based on your provided data
data = {
    "City": ["Acton", "Agoura Hills", "Agoura Hills(PO Boxes)", "Agua Dulce", "Alhambra"],
    "Zip_Code": [93510, 91301, 91376, 91390, 91801]
}

df = pd.DataFrame(data)

# Load the GeoJSON file for zip codes in Los Angeles
# You can find GeoJSON files for zip codes online, or use a public dataset
# For example, you can use the following GeoJSON file for California zip codes:
# https://github.com/OpenDataDE/State-zip-code-GeoJSON/blob/master/ca_california_zip_codes_geo.min.json

# For simplicity, let's assume you have a GeoJSON file for LA zip codes
geojson_url = "https://raw.githubusercontent.com/OpenDataDE/State-zip-code-GeoJSON/master/ca_california_zip_codes_geo.min.json"

# Create the choropleth map
fig = px.choropleth(df, 
                    geojson=geojson_url, 
                    locations='Zip_Code', 
                    color='Zip_Code',
                    scope="usa",
                    featureidkey="properties.ZCTA5CE10",
                    labels={'Zip_Code':'Zip Code'},
                    title="Los Angeles Zip Code Choropleth Map")

# Update layout for better visualization
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

# Show the map
fig.show()

Note: doesn't explain the dataset or better represent anything.

add year elements

In [ ]:
df2_filtered['start_year'] = df2_filtered['LOCATION START DATE'].dt.year
df2_filtered['start_month'] = df2_filtered['LOCATION START DATE'].dt.month

# If LOCATION END DATE is null, the business is still open (Yes); otherwise, it's closed (No)
df2_filtered['is_open'] = np.where(df2_filtered['LOCATION END DATE'].isnull(), 'Yes', 'No')

download_date = pd.to_datetime("20250202", format="%Y%m%d")
# (31*7 + 30*4 +28) / 12 = 30.4
df2_filtered['duration'] = (df2_filtered['LOCATION END DATE'].fillna(download_date) - df2_filtered['LOCATION START DATE']).dt.days / 30.4

In [ ]:
# df2_filtered[df2_filtered['is_open'] == 'No'].head(5)
df2_filtered.head(5)

In [ ]:
# save dataset clean up progress
df2_filtered.to_csv('dataset\\business_filtered.csv', index=False)

### EDA: append info from NAICS csv file

In [ ]:
import pandas as pd
df_cleaned = pd.read_csv('dataset\\business_filtered.csv')

In [ ]:
df_naics = pd.read_csv('dataset\\naics_2_clean.csv')
code_sector_dict = df_naics.set_index('Code')['Sector_Title'].to_dict()
code_sector_dict

In [ ]:
df_cleaned['NAICS_2'] = df_cleaned['NAICS'].astype(str).str[:2]
df_cleaned['NAICS_2'] = df_cleaned['NAICS_2'].astype(int)

In [ ]:
df_cleaned['NAICS_2_Title'] = df_cleaned['NAICS_2'].map(code_sector_dict)

In [ ]:
df_cleaned.info()

**Note**: later interactive map work will toward app engine `app_geomap.py` 

### dataset modify 

In [ ]:
url = 'https://media.githubusercontent.com/media/EricSJSU-DataScience/CS163_project/refs/heads/main/dataset/business_filtered.csv'
file_path = 'dataset\\business_filtered.csv'
df = pd.read_csv(file_path)

In [ ]:
df['is_open'] = df['is_open'].map({'Yes': True, 'No': False})

In [ ]:
df.to_csv('dataset\\business_filtered.csv', index=False)